# Vast.ai: serve Qwen3.8-27B with Ollama

This notebook does one thing: rent a Vast.ai GPU instance, start the official `ollama/ollama` image, pull `qwen3.8:27b-q4_K_M`, and call its OpenAI-compatible API through a private SSH tunnel.

The flow follows the official [Vast.ai instance CLI](https://docs.vast.ai/cli/reference/create-instance), [Ollama Docker](https://docs.ollama.com/docker), [pull API](https://docs.ollama.com/api/pull), and [OpenAI compatibility](https://docs.ollama.com/api/openai-compatibility) documentation.

> **Cost:** creating an instance starts billing. Run the cleanup cell when finished.
>
> **Security:** Ollama's local API has no authentication; the `api_key="ollama"` value used later is only an OpenAI-SDK placeholder and provides no protection. This deployment is protected by its network and SSH configuration:
>
> - Ollama listens only on `127.0.0.1:11434` inside the container, and the notebook does not publish that port. Someone cannot use Ollama merely by knowing the Vast.ai public IP and SSH port.
> - The only access path is an encrypted SSH tunnel authenticated with `VASTAI_SSH_KEY_PATH`; Vast.ai SSH accepts keys rather than passwords. Keep the private key secret. So this method is secure.
> - The tunnel's local endpoint also binds to `127.0.0.1`, so it is not reachable from other computers. Processes or users already on your local computer may be able to use it while the tunnel is open.
> - The underlying Vast.ai provider technically controls the host. For sensitive prompts or data, prefer a Secure Cloud offer (`datacenter=true`), encrypt sensitive data, avoid leaving credentials on the instance, and destroy the instance when finished. `verified=true` alone is not the same as Secure Cloud.
> - Do not add `-p 11434:11434`, change `OLLAMA_HOST` to `0.0.0.0`, or bind the local tunnel to `0.0.0.0` unless you also add proper authentication and firewall restrictions.

## 1. Setup

Create the environment from this folder and select its Jupyter kernel:

```bash
uv sync --locked
uv run python -m ipykernel install --user --name gpu-deployments --display-name gpu-deployments
```

Required in `.env`:

- `VAST_API_KEY`: Vast.ai account API key.
- `VASTAI_SSH_KEY_PATH`: private key whose public half is already registered in Vast.ai.
- `OLLAMA_MODEL` is optional; the default is `qwen3.8:27b-q4_K_M` (18 GB).

In [ ]:
import json
import os
import shlex
import socket
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

VAST_API_KEY = os.getenv("VAST_API_KEY", "").strip()
SSH_KEY_PATH = Path(os.path.expanduser(os.getenv("VASTAI_SSH_KEY_PATH", "~/.ssh/id_vastai_ed25519")))
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "").strip() or "qwen3.8:27b-q4_K_M"

if not VAST_API_KEY:
    raise RuntimeError("Set VAST_API_KEY in .env")
if not SSH_KEY_PATH.is_file():
    raise RuntimeError(f"SSH private key not found: {SSH_KEY_PATH}")

INSTANCE_ID = None
TUNNEL = None
print(f"Model: {OLLAMA_MODEL}")
print(f"SSH key: {SSH_KEY_PATH}")

In [ ]:
def run_vastai(args: list[str], *, json_output: bool = True):
    """Run Vast.ai without putting the API key in notebook output or argv."""
    cmd = ["vastai", *args]
    if json_output:
        cmd.append("--raw")
    print("$", shlex.join(cmd))
    env = os.environ.copy()
    env["VAST_API_KEY"] = VAST_API_KEY
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout).strip())
    if not json_output:
        if result.stdout.strip():
            print(result.stdout.strip())
        return None
    return json.loads(result.stdout) if result.stdout.strip() else None


def as_list(value, key: str) -> list[dict]:
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        items = value.get(key, [])
        return items if isinstance(items, list) else list(items.values())
    return []


user = run_vastai(["show", "user"])
print(f"Authenticated as Vast.ai user {user.get('id', '<unknown>')}")

## 2. Find a suitable offer

The query asks for two RTX 3090 GPUs (48 GB total VRAM), at least 80 GB of disk, reliable networking, and a verified on-demand host. The Vast.ai CLI query language reports `gpu_ram` in GB (the REST API uses MB). Adjust the country list if no offer is available.

In [ ]:
EUROPE = ["DE", "FR", "GB", "NL", "SE", "NO", "FI", "DK", "PL", "ES", "IT", "PT", "CH", "AT", "BE", "IE", "CZ"]
OFFER_QUERY = (
    "gpu_name = RTX_3090 num_gpus = 2 gpu_ram >= 24 "
    "disk_space >= 80 direct_port_count >= 1 reliability >= 0.98 "
    #f"geolocation in [{','.join(EUROPE)}] datacenter = true verified = true rentable = true"
    f"geolocation in [{','.join(EUROPE)}] verified = true rentable = true"
)

offers = as_list(run_vastai(["search", "offers", OFFER_QUERY, "--order", "dph_total"]), "offers")
if not offers:
    raise RuntimeError("No matching offer. Widen EUROPE or relax reliability/verified, then rerun.")

for offer in offers[:5]:
    print(
        f"offer={offer['id']}  {offer.get('gpu_name')} x{offer.get('num_gpus')}  "
        f"VRAM={offer.get('gpu_total_ram', '?')} MB  ${offer.get('dph_total', 0):.3f}/hour  "
        f"{offer.get('geolocation', '?')}"
    )

OFFER_ID = offers[0]["id"]
print(f"Selected cheapest offer: {OFFER_ID}")

## 3. Create the instance

`--ssh --direct` is essential: it tells Vast.ai to use its SSH runtime and provision the connection that the tunnel needs. The previous notebook omitted these flags. The on-start command only starts Ollama; the model pull happens later through Ollama's API so progress and errors remain visible here.

In [ ]:
ONSTART = (
    "mkdir -p /workspace/ollama-models; "
    "OLLAMA_HOST=127.0.0.1:11434 "
    "OLLAMA_MODELS=/workspace/ollama-models "
    "nohup ollama serve >/tmp/ollama-serve.log 2>&1 &"
)

created = run_vastai([
    "create", "instance", str(OFFER_ID),
    "--image", "ollama/ollama:latest",
    "--disk", "80",
    "--label", "ollama-qwen3.8-27b",
    "--ssh",
    "--direct",
    "--onstart-cmd", ONSTART,
])
INSTANCE_ID = int(created["new_contract"])
print(f"Created instance {INSTANCE_ID}. Billing is now active.")

### Emergency cleanup

If any later cell fails, this works independently after you set the numeric ID.

In [ ]:
def destroy_instance(instance_id: int) -> None:
    run_vastai(["destroy", "instance", str(instance_id), "-y"], json_output=False)

# Example after a failure or kernel restart:
# destroy_instance(12345678)

## 4. Wait for SSH and open the private tunnel

The tunnel maps local `127.0.0.1:<free port>` to Ollama's in-container `127.0.0.1:11434`. Nothing is published on the instance's public interface.

In [ ]:
def wait_for_running(instance_id: int, timeout_s: int = 600) -> dict:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        info = run_vastai(["show", "instance", str(instance_id)])
        if info.get("error"):
            raise RuntimeError(info.get("msg", info))
        status = info.get("actual_status", "unknown")
        print(f"instance {instance_id}: {status}")
        if status == "running":
            return info
        time.sleep(10)
    raise TimeoutError(f"Instance {instance_id} did not reach running state")


def ssh_endpoint(instance_id: int) -> tuple[str, int]:
    env = os.environ.copy()
    env["VAST_API_KEY"] = VAST_API_KEY
    result = subprocess.run(
        ["vastai", "ssh-url", str(instance_id)], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout).strip())
    url = result.stdout.strip()
    parsed = urllib.parse.urlparse(url if "://" in url else f"ssh://{url}")
    if not parsed.hostname or not parsed.port:
        raise RuntimeError(f"Could not parse Vast.ai SSH URL: {url!r}")
    return parsed.hostname, parsed.port


def free_local_port() -> int:
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def open_tunnel(instance_id: int, timeout_s: int = 300) -> tuple[subprocess.Popen, int]:
    deadline = time.time() + timeout_s
    last_error = "SSH endpoint not ready"
    while time.time() < deadline:
        try:
            host, port = ssh_endpoint(instance_id)
            local_port = free_local_port()
            cmd = [
                "ssh", "-i", str(SSH_KEY_PATH), "-N",
                "-L", f"127.0.0.1:{local_port}:127.0.0.1:11434",
                "-o", "BatchMode=yes", "-o", "IdentitiesOnly=yes",
                "-o", "ExitOnForwardFailure=yes",
                "-o", "StrictHostKeyChecking=accept-new",
                "-o", "ConnectTimeout=15", "-o", "ServerAliveInterval=30",
                "-p", str(port), f"root@{host}",
            ]
            print("$", shlex.join(cmd))
            process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
            time.sleep(3)
            if process.poll() is None:
                return process, local_port
            last_error = (process.stderr.read() or "SSH exited").strip()
        except Exception as exc:
            last_error = str(exc)
        print(f"SSH not ready: {last_error[:160]}")
        time.sleep(10)
    raise TimeoutError(f"Could not open SSH tunnel: {last_error}")


wait_for_running(INSTANCE_ID)
TUNNEL, LOCAL_PORT = open_tunnel(INSTANCE_ID)
OLLAMA_URL = f"http://127.0.0.1:{LOCAL_PORT}"
print(f"Private Ollama endpoint: {OLLAMA_URL}")

## 5. Wait for Ollama and pull the model

The official `POST /api/pull` endpoint streams newline-delimited JSON. This cell shows status and percentage and raises on an Ollama error, including errors sent after the HTTP stream has already started.

In [ ]:
def wait_for_ollama(base_url: str, timeout_s: int = 180) -> None:
    deadline = time.time() + timeout_s
    last_error = "not reachable"
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f"{base_url}/api/tags", timeout=10) as response:
                json.load(response)
            print("Ollama is ready.")
            return
        except Exception as exc:
            last_error = str(exc)
            time.sleep(5)
    raise TimeoutError(
        f"Ollama did not start: {last_error}. SSH in and inspect /tmp/ollama-serve.log."
    )


def pull_model(base_url: str, model: str) -> None:
    body = json.dumps({"model": model, "stream": True}).encode()
    request = urllib.request.Request(
        f"{base_url}/api/pull", data=body, headers={"Content-Type": "application/json"}
    )
    last_status = None
    last_bucket = -1
    try:
        with urllib.request.urlopen(request, timeout=300) as response:
            for raw_line in response:
                event = json.loads(raw_line)
                if event.get("error"):
                    raise RuntimeError(event["error"])
                status = event.get("status", "working")
                total = event.get("total") or 0
                completed = event.get("completed") or 0
                percent = int(completed * 100 / total) if total else None
                bucket = percent // 5 if percent is not None else -1
                if status != last_status or bucket != last_bucket:
                    suffix = f" {percent}%" if percent is not None else ""
                    print(f"{status}{suffix}")
                    last_status, last_bucket = status, bucket
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode(errors="replace")
        raise RuntimeError(f"Ollama pull failed ({exc.code}): {detail}") from exc


def show_model(base_url: str, model: str) -> dict:
    request = urllib.request.Request(
        f"{base_url}/api/show",
        data=json.dumps({"model": model, "verbose": False}).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)


wait_for_ollama(OLLAMA_URL)
pull_model(OLLAMA_URL, OLLAMA_MODEL)
model_info = show_model(OLLAMA_URL, OLLAMA_MODEL)
print("Pulled:", model_info.get("details", {}))

## 6. Serve and test Qwen3.8

Ollama implements `/v1/chat/completions`. The OpenAI SDK requires an API-key string, but Ollama ignores it for the local API; access is protected by the SSH tunnel.

In [ ]:
PROMPT = """
Write a Python function:

```python
def smallest_missing_positive(numbers: list[int]) -> int:
```

Given an unsorted list of integers, return the smallest positive integer that does not occur in the list.

Requirements:

- Run in `O(n)` time.
- Use `O(1)` additional space.
- Modify the input list if necessary.
- Do not use `set()`, sorting, or additional lists.
- Explain the algorithm briefly.
- Include tests for empty input, duplicates, negative numbers, and already consecutive values.

Examples:

```python
smallest_missing_positive([3, 4, -1, 1]) == 2
smallest_missing_positive([1, 2, 0]) == 3
smallest_missing_positive([7, 8, 9, 11, 12]) == 1
smallest_missing_positive([1, 1, 2, 2]) == 3
```
"""

In [ ]:
client = OpenAI(base_url=f"{OLLAMA_URL}/v1/", api_key="ollama")

started = time.perf_counter()
response = client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": PROMPT}],
    reasoning_effort="none",
    #max_tokens=40,
)
elapsed = time.perf_counter() - started
print(response.choices[0].message.content)
print(f"\n\nLatency including first model load: {elapsed:.1f}s")

with urllib.request.urlopen(f"{OLLAMA_URL}/api/ps", timeout=10) as result:
    running = json.load(result).get("models", [])
print("Running model details:", json.dumps(running, indent=2)[:1200])

### 6.1 Measure tokens per second

Two measurements are useful:

- **End-to-end throughput** uses the OpenAI-compatible response's output-token count divided by wall-clock time. It includes model loading, prompt evaluation, SSH/network overhead, and generation, so it is useful for user-visible performance but understates pure generation speed.
- **Generation throughput** uses Ollama's native `eval_count / eval_duration`. This isolates token generation and is the better value for comparing GPU offers. Ollama reports durations in nanoseconds.

The native call below is deliberately a second, warm request: the first request above has already loaded the model.

In [ ]:
# Approximate end-to-end throughput from the OpenAI-compatible call above.
if response.usage is None or response.usage.completion_tokens is None:
    print("The OpenAI-compatible response did not include token usage.")
else:
    output_tokens = response.usage.completion_tokens
    end_to_end_tok_s = output_tokens / elapsed
    print(f"OpenAI-compatible output tokens: {output_tokens}")
    print(f"End-to-end throughput: {end_to_end_tok_s:.2f} tok/s")

# Accurate warm generation throughput from Ollama's native timing fields.
payload = {
    "model": OLLAMA_MODEL,
    "messages": [{"role": "user", "content": PROMPT}],
    "stream": False,
}
request = urllib.request.Request(
    f"{OLLAMA_URL}/api/chat",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)

with urllib.request.urlopen(request, timeout=600) as result:
    metrics = json.load(result)

eval_count = metrics.get("eval_count", 0)
eval_duration_s = metrics.get("eval_duration", 0) / 1e9
generation_tok_s = eval_count / eval_duration_s if eval_duration_s else 0.0

prompt_count = metrics.get("prompt_eval_count", 0)
cached_count = metrics.get("prompt_eval_cached_count", 0)
uncached_prompt_count = max(prompt_count - cached_count, 0)
prompt_duration_s = metrics.get("prompt_eval_duration", 0) / 1e9
prompt_tok_s = uncached_prompt_count / prompt_duration_s if prompt_duration_s else 0.0

print("\nWarm native Ollama response:\n")
print(metrics["message"]["content"])
print(f"\nGenerated tokens: {eval_count}")
print(f"Generation speed: {generation_tok_s:.2f} tok/s")
print(f"Uncached prompt speed: {prompt_tok_s:.2f} tok/s")
print(f"Model load: {metrics.get('load_duration', 0) / 1e9:.2f}s")
print(f"Total native request: {metrics.get('total_duration', 0) / 1e9:.2f}s")

## 7. Cleanup

This closes the local tunnel and destroys the billed Vast.ai instance. Verify the instance is gone in the Vast.ai console.

In [ ]:
if TUNNEL is not None and TUNNEL.poll() is None:
    TUNNEL.terminate()
    TUNNEL.wait(timeout=10)
    print("SSH tunnel closed.")

if INSTANCE_ID is not None:
    destroy_instance(INSTANCE_ID)
    print(f"Destroyed instance {INSTANCE_ID}.")